# Propiedades geométricas de contornos

**Duración estimada:** 65 minutos

## Objetivo

Este cuaderno vuelve a un enfoque más controlado y didáctico para estudiar propiedades geométricas. Primero trabajamos con formas simples sobre un lienzo binario y después, si querés, podés transferir la idea a imágenes reales.

## Resultados de aprendizaje

Al final vas a poder:

- calcular área y perímetro;
- obtener centroide y caja contenedora;
- usar `approxPolyDP()` para resumir una forma;
- interpretar qué dice cada medida sobre el objeto observado.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


CARPETA_IMAGENES = Path("Imagenes")


def abrir_imagen_bgr(nombre_archivo):
    """Abre una imagen en color usando el orden BGR de OpenCV."""
    ruta = CARPETA_IMAGENES / nombre_archivo
    imagen_bgr = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    if imagen_bgr is None:
        raise FileNotFoundError(f"No pude abrir la imagen: {ruta}")
    return imagen_bgr


def abrir_imagen_rgb(nombre_archivo):
    """Abre una imagen y la convierte a RGB para mostrarla con Matplotlib."""
    imagen_bgr = abrir_imagen_bgr(nombre_archivo)
    imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    return imagen_rgb


def preparar_eje_imagen(eje, imagen, titulo, mapa=None):
    """Muestra una imagen conservando ejes y coordenadas visibles."""
    eje.imshow(imagen, cmap=mapa)
    eje.set_title(titulo, loc="left", fontweight="bold")
    eje.set_xlabel("x (columnas)")
    eje.set_ylabel("y (filas)")

    if imagen.ndim == 2:
        alto, ancho = imagen.shape
    else:
        alto, ancho = imagen.shape[:2]

    paso_x = max(50, ancho // 6)
    paso_y = max(50, alto // 6)
    eje.set_xticks(np.arange(0, ancho + 1, paso_x))
    eje.set_yticks(np.arange(0, alto + 1, paso_y))
    eje.grid(color="white", alpha=0.25, linewidth=0.6)


def mostrar_una_imagen(imagen, titulo, mapa=None, tamano=(8, 6)):
    fig, eje = plt.subplots(figsize=tamano, constrained_layout=True)
    preparar_eje_imagen(eje, imagen, titulo, mapa)
    plt.show()


def mostrar_varias_imagenes(imagenes, titulos, mapas=None, tamano=(15, 5)):
    if mapas is None:
        mapas = [None] * len(imagenes)

    fig, ejes = plt.subplots(1, len(imagenes), figsize=tamano, constrained_layout=True)
    if len(imagenes) == 1:
        ejes = [ejes]

    for eje, imagen, titulo, mapa in zip(ejes, imagenes, titulos, mapas):
        preparar_eje_imagen(eje, imagen, titulo, mapa)

    plt.show()


def mostrar_histograma_gris(imagen_gris, titulo):
    histograma, bordes = np.histogram(imagen_gris.ravel(), bins=256, range=(0, 256))
    plt.figure(figsize=(9, 4))
    plt.plot(bordes[:-1], histograma, color="black")
    plt.title(titulo, loc="left", fontweight="bold")
    plt.xlabel("Intensidad")
    plt.ylabel("Cantidad de píxeles")
    plt.grid(alpha=0.3)
    plt.xlim(0, 255)
    plt.show()


def mostrar_histogramas_bgr(imagen_bgr, titulo_general):
    nombres = ["azul", "verde", "rojo"]
    colores = ["tab:blue", "tab:green", "tab:red"]
    fig, ejes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

    for indice in range(3):
        histograma = cv2.calcHist([imagen_bgr], [indice], None, [256], [0, 256]).ravel()
        ejes[indice].plot(histograma, color=colores[indice])
        ejes[indice].set_title(f"Canal {nombres[indice]}", loc="left", fontweight="bold")
        ejes[indice].set_xlabel("Intensidad")
        ejes[indice].set_ylabel("Frecuencia")
        ejes[indice].grid(alpha=0.25)

    fig.suptitle(titulo_general, x=0.01, ha="left", fontweight="bold")
    plt.show()

# Creamos un lienzo negro y dibujamos tres formas simples en blanco.
alto_lienzo = 320
ancho_lienzo = 520
lienzo = np.zeros((alto_lienzo, ancho_lienzo), dtype=np.uint8)

# Rectángulo.
cv2.rectangle(lienzo, (30, 60), (170, 250), 255, -1)

# Círculo.
cv2.circle(lienzo, (290, 160), 70, 255, -1)

# Triángulo.
vertices_triangulo = np.array([[400, 250], [500, 250], [450, 80]], dtype=np.int32)
cv2.fillPoly(lienzo, [vertices_triangulo], 255)

mostrar_una_imagen(lienzo, "Lienzo binario con tres formas", mapa="gray", tamano=(10, 6))


## 1. Por qué conviene empezar con formas controladas

Cuando el objetivo es aprender qué mide cada función, una escena controlada ayuda muchísimo. Acá no hay ruido, textura ni sombras: solo formas blancas sobre fondo negro. Eso permite que el foco esté en la geometría y no en problemas de segmentación previos.


In [ ]:
contornos, _ = cv2.findContours(lienzo.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos = sorted(contornos, key=cv2.contourArea, reverse=True)


def medir_contorno(contorno):
    """Calcula propiedades geométricas básicas de un contorno."""
    area = cv2.contourArea(contorno)
    perimetro = cv2.arcLength(contorno, True)
    x, y, ancho, alto = cv2.boundingRect(contorno)

    momentos = cv2.moments(contorno)
    if momentos["m00"] != 0:
        centro_x = int(momentos["m10"] / momentos["m00"])
        centro_y = int(momentos["m01"] / momentos["m00"])
    else:
        centro_x = -1
        centro_y = -1

    epsilon = 0.03 * perimetro
    aproximacion = cv2.approxPolyDP(contorno, epsilon, True)

    return {
        "area": area,
        "perimetro": perimetro,
        "caja": (x, y, ancho, alto),
        "centroide": (centro_x, centro_y),
        "vertices_aproximados": len(aproximacion),
    }


mediciones = []
for contorno in contornos:
    mediciones.append(medir_contorno(contorno))

imagen_anotada = cv2.cvtColor(lienzo, cv2.COLOR_GRAY2RGB)

for indice, contorno in enumerate(contornos, start=1):
    medicion = mediciones[indice - 1]
    x, y, ancho, alto = medicion["caja"]
    centro_x, centro_y = medicion["centroide"]

    cv2.drawContours(imagen_anotada, [contorno], -1, (255, 140, 0), 2)
    cv2.rectangle(imagen_anotada, (x, y), (x + ancho, y + alto), (0, 255, 0), 2)
    cv2.circle(imagen_anotada, (centro_x, centro_y), 4, (255, 255, 0), -1)
    cv2.putText(imagen_anotada, f"F{indice}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

mostrar_una_imagen(imagen_anotada, "Formas con contornos, caja y centroide", tamano=(10, 6))


In [ ]:
for indice, medicion in enumerate(mediciones, start=1):
    print(f"Forma {indice}")
    print(f"  Área: {medicion['area']:.1f} píxeles cuadrados")
    print(f"  Perímetro: {medicion['perimetro']:.1f} píxeles")
    print(f"  Caja contenedora: {medicion['caja']}")
    print(f"  Centroide: {medicion['centroide']}")
    print(f"  Vértices aproximados: {medicion['vertices_aproximados']}")


## 2. Cómo leer estas medidas

- **Área:** cuánto ocupa la forma.
- **Perímetro:** cuánto mide el borde.
- **Caja contenedora:** el rectángulo más simple que la encierra.
- **Centroide:** una forma de ubicar el centro geométrico.
- **Vértices aproximados:** una pista útil para resumir la geometría.

Este enfoque es más útil para aprender porque cada medida se ve con claridad sobre formas conocidas.


In [ ]:
# Transferencia opcional a una imagen real: usamos un contorno del cuaderno anterior.
imagen_globos_bgr = abrir_imagen_bgr("globos.jpg")
imagen_globos_rgb = cv2.cvtColor(imagen_globos_bgr, cv2.COLOR_BGR2RGB)
imagen_globos_hsv = cv2.cvtColor(imagen_globos_bgr, cv2.COLOR_BGR2HSV)

mascara_naranja = cv2.inRange(imagen_globos_hsv, np.array([8, 80, 80]), np.array([22, 255, 255]))
kernel = np.ones((5, 5), np.uint8)
mascara_naranja = cv2.morphologyEx(mascara_naranja, cv2.MORPH_OPEN, kernel, iterations=1)
mascara_naranja = cv2.morphologyEx(mascara_naranja, cv2.MORPH_CLOSE, kernel, iterations=2)

contornos_reales, _ = cv2.findContours(mascara_naranja.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

contorno_principal = None
area_mayor = 0
for contorno in contornos_reales:
    area = cv2.contourArea(contorno)
    if area > area_mayor:
        area_mayor = area
        contorno_principal = contorno

if contorno_principal is not None:
    medicion_real = medir_contorno(contorno_principal)
    imagen_real_anotada = imagen_globos_rgb.copy()
    x, y, ancho, alto = medicion_real["caja"]
    centro_x, centro_y = medicion_real["centroide"]
    cv2.drawContours(imagen_real_anotada, [contorno_principal], -1, (255, 140, 0), 3)
    cv2.rectangle(imagen_real_anotada, (x, y), (x + ancho, y + alto), (0, 255, 0), 2)
    cv2.circle(imagen_real_anotada, (centro_x, centro_y), 5, (255, 255, 255), -1)

    mostrar_una_imagen(imagen_real_anotada, "Transferencia a una imagen real de globos")

    print("Medición del contorno principal en la imagen real:")
    print(medicion_real)


## Cierre

Este cuaderno recupera una decisión didáctica importante: para aprender geometría conviene empezar con un caso claro y controlado. Después sí podemos transferir esa lógica a imágenes reales, donde aparecen ruido, sombras y regiones fusionadas.
